# HUẤN LUYỆN CẶP THỬ NGHIỆM M3 (ASTNN) TRÊN BIGCLONEBENCH JAVA

Notebook này thực hiện huấn luyện **Cặp 2 kịch bản tinh chỉnh độc lập** cho mô hình **M3** theo nguyên tắc OFAT:
1. **Kịch bản 1:** `EXP-M3-EP` — Tăng Epochs lên 20 (Early Stopping pat=5) (File cấu hình: `configs/tuning/exp_m3_ep.yaml`)
2. **Kịch bản 2:** `EXP-M3-LR` — Hạ Learning Rate xuống 1e-4 (File cấu hình: `configs/tuning/exp_m3_lr.yaml`)

### Cơ chế An toàn & Tự động dọn dẹp:
- Tự động dò tìm dataset trong `/kaggle/input` mà không phụ thuộc username.
- Chạy lần lượt từng kịch bản kèm logging chi tiết và lưu checkpoint an toàn.
- **Auto-Cleanup Cuối cùng:** Tự động xóa sạch toàn bộ mã nguồn `src/`, `scripts/`, `data/`, chỉ xuất ra các file `.pt`, `.json`, `.csv` và file nén kết quả tinh gọn.

---

In [ ]:
# 1. THIẾT LẬP MÔI TRƯỜNG & HÀM BỔ TRỢ STREAM LOG AN TOÀN
import base64, io, json, os, shutil, subprocess, sys, tarfile, time, yaml
from pathlib import Path

def sh(cmd, **kw):
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_file = Path("/kaggle/working/live_progress.log")
    with log_file.open("a", encoding="utf-8") as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Lệnh thất bại (rc={rc}): {cmd}")
    return ""

print("Python:", sys.version.split()[0], flush=True)
print("Thư mục làm việc hiện tại:", os.getcwd(), flush=True)

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN DỰ ÁN VÀO THƯ MỤC RIÊNG
WORK_DIR = Path("/kaggle/working/project")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = "..."

print(f"Giải nén mã nguồn ({len(BUNDLE_B64)/1024:.1f} KB) vào {WORK_DIR}...")
buf = io.BytesIO(base64.b64decode(BUNDLE_B64))
with tarfile.open(fileobj=buf, mode="r:gz") as tar:
    tar.extractall(WORK_DIR)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

assert Path('configs/tuning/exp_m3_ep.yaml').exists(), f'Khong tim thay file configs/tuning/exp_m3_ep.yaml'
assert Path('configs/tuning/exp_m3_lr.yaml').exists(), f'Khong tim thay file configs/tuning/exp_m3_lr.yaml'
print('Da giai nen ma nguon va xac nhan day du file cau hinh thanh cong!')

In [ ]:
# 3. CÀI ĐẶT DEPENDENCIES & XÁC THỰC GPU BẮT BUỘC
sh("pip install -q --no-input 'tree-sitter>=0.23.0' 'tree-sitter-java>=0.23.5' 'tree-sitter-cpp>=0.23.0' 'datasets>=2.14.0' 'transformers>=4.30.0' 'torch_geometric' 'pyarrow' 'tqdm' 'pyyaml' 'scikit-learn'")

import torch
if not torch.cuda.is_available():
    raise SystemError('GPU KHONG KHA DUNG! Hay vao menu Settings -> Accelerator -> Chon GPU (T4/P100) truoc khi chay.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f'>>> GPU HOAT DONG: {gpu_name} | VRAM: {vram_gb} GB <<<')
sh('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv')

In [ ]:
# 4. TỰ ĐỘNG DÒ TÌM VÀ LIÊN KẾT ĐẶC TRƯNG TỪ DATASET KAGGLE
import numpy as np

# Tim kiem de quy moi thu muc con trong /kaggle/input de tim du lieu phu hop bat ke username
ds_root = None
input_path = Path('/kaggle/input')
if input_path.exists():
    for p in input_path.rglob('*'):
        if p.is_dir() and ('semanticcodeclone' in p.name.lower() or 'bigclonebench' in p.name.lower()):
            ds_root = p
            break
        if p.name in ('bcb_ast_cache', 'bcb_cpg_export', 'bcb_ast_cache.npz', 'bcb_cpg_export.tar.gz'):
            ds_root = p.parent
            break

print(f'Thu muc dataset phat hien: {ds_root}')
if ds_root:
    sh(f'ls -la {ds_root}/')
# Lien ket AST Cache cho M3
os.makedirs('artifacts/bcb_features', exist_ok=True)
ast_target = Path('artifacts/bcb_features/bcb_ast_cache.npz')
ast_src = ds_root / 'bcb_ast_cache' if ds_root else None
if ast_src and ast_src.exists():
    npy_files = list(ast_src.glob('*.npy'))
    if npy_files:
        print(f'Dong goi {len(npy_files)} file AST .npy vao {ast_target.name}...')
        arrays = {f.stem: np.load(f, allow_pickle=True) for f in npy_files}
        np.savez_compressed(ast_target, **arrays)
        print(f'Tao thanh cong {ast_target} ({ast_target.stat().st_size / 1024:.1f} KB)!')
elif (ds_root / 'bcb_ast_cache.npz').exists() if ds_root else False:
    sh(f'cp {ds_root}/bcb_ast_cache.npz {ast_target}')
    print(f'Da sao chep {ast_target} tu dataset!')
else:
    print('Thong bao: Chua co AST cache co san, se tu dong parse online.')


In [ ]:
# 5. TẢI CÁC SPLIT DỮ LIỆU CHUẨN (TRAIN 30k, VAL 3k, TEST 5k CÂN BẰNG 50:50)
data_raw = Path('data/raw/bigclonebench')
data_raw.mkdir(parents=True, exist_ok=True)

# Kiem tra neu dataset da co san file parquet thi copy nhanh de tiet kiem thoi gian
cached_parquet = ds_root / 'bcb_train.parquet' if ds_root else None
if cached_parquet and cached_parquet.exists():
    print('Phat hien du lieu parquet san co tu dataset, sao chep truc tiep...')
    sh(f'cp {ds_root}/*.parquet data/raw/bigclonebench/')
else:
    print('Tai truc tiep cac split BigCloneBench tu Hugging Face...')
    sh('python scripts/download_bcb_subsets.py --splits-custom --train-samples 30000 --val-samples 3000 --test-samples 5000')

sh('ls -lh data/raw/bigclonebench/*.parquet')
assert (data_raw / 'bcb_train.parquet').exists(), 'bcb_train.parquet phai ton tai truoc khi train!'

In [ ]:
# 6. HUẤN LUYỆN & ĐÁNH GIÁ KỊCH BẢN 1: EXP-M3-EP (Tăng Epochs lên 20 (Early Stopping pat=5))
print('=' * 70)
print('>>> BAT DAU KICH BAN 1: EXP-M3-EP (Tăng Epochs lên 20 (Early Stopping pat=5)) <<<')
print('=' * 70)
cfg1 = 'configs/tuning/exp_m3_ep.yaml'
t1_start = time.perf_counter()

# Preflight check 1
sh(f'python -m src.experiments.bcb_runner --config {cfg1} --mode preflight --device cuda')

# Train 1
sh(f'python -m src.experiments.bcb_runner --config {cfg1} --mode train --device cuda --time-budget-s 7200')

# Evaluate 1
sh(f'python -m src.experiments.bcb_runner --config {cfg1} --mode evaluate --device cuda')

print(f'>>> HOAN TAT EXP-M3-EP trong {(time.perf_counter() - t1_start)/60:.1f} phut! <<<')

In [ ]:
# 7. HUẤN LUYỆN & ĐÁNH GIÁ KỊCH BẢN 2: EXP-M3-LR (Hạ Learning Rate xuống 1e-4)
print('=' * 70)
print('>>> BAT DAU KICH BAN 2: EXP-M3-LR (Hạ Learning Rate xuống 1e-4) <<<')
print('=' * 70)
cfg2 = 'configs/tuning/exp_m3_lr.yaml'
t2_start = time.perf_counter()

# Preflight check 2
sh(f'python -m src.experiments.bcb_runner --config {cfg2} --mode preflight --device cuda')

# Train 2
sh(f'python -m src.experiments.bcb_runner --config {cfg2} --mode train --device cuda --time-budget-s 7200')

# Evaluate 2
sh(f'python -m src.experiments.bcb_runner --config {cfg2} --mode evaluate --device cuda')

print(f'>>> HOAN TAT EXP-M3-LR trong {(time.perf_counter() - t2_start)/60:.1f} phut! <<<')

In [ ]:
# 8. DỌN DẸP SẠCH SẼ (AUTO-CLEANUP) & CHỈ XUẤT .PT, .JSON, .CSV TINH GỌN
print('=' * 70)
print('TIẾN HÀNH THU THẬP KẾT QUẢ & DỌN DẸP SẠCH MÃ NGUỒN KHÔNG LIÊN QUAN...')
print('=' * 70)

WORKING_DIR = Path('/kaggle/working')
CLEAN_DIR = WORKING_DIR / 'clean_results'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# 1. Quet va sao chep toan bo file quan trong (.pt, .json, .csv) tu results/
results_dir = Path('/kaggle/working/project/results')
copied_files = []
if results_dir.exists():
    for p in results_dir.rglob('*'):
        if p.is_file() and p.suffix in ('.pt', '.json', '.csv'):
            # Dat ten ro rang theo thu muc cha: vd exp_m3_ep_best_checkpoint.pt
            subfolder = p.parent.name
            out_name = f"{subfolder}_{p.name}" if not p.name.startswith(subfolder) else p.name
            dest_clean = CLEAN_DIR / out_name
            shutil.copy2(p, dest_clean)
            # Sao chep truc tiep ra /kaggle/working de hien thi ngay tren Kaggle UI Output
            shutil.copy2(p, WORKING_DIR / out_name)
            copied_files.append(out_name)

# Sao chep live_progress.log neu co
if (WORKING_DIR / 'live_progress.log').exists():
    shutil.copy2(WORKING_DIR / 'live_progress.log', CLEAN_DIR / 'live_progress.log')

# 2. Dong goi toan bo ket qua sach vao 1 file tar.gz duy nhat
tar_name = 'm3_tuning_clean_bundle.tar.gz'
tar_path = WORKING_DIR / tar_name
with tarfile.open(tar_path, 'w:gz') as tar:
    for f in CLEAN_DIR.glob('*'):
        tar.add(f, arcname=f.name)
print(f'>>> Da tao file goi ket qua: {tar_name} ({tar_path.stat().st_size / 1024 / 1024:.2f} MB) <<<')

# 3. XOA TRIET DE MA NGUON, CACHE VA DATA TRUOC KHI THOAT
os.chdir('/kaggle/working')
junk_paths = [
    Path('/kaggle/working/project'),
    Path('/kaggle/working/data'),
    Path('/kaggle/working/artifacts'),
    Path('/kaggle/working/clean_results'),
]
for jp in junk_paths:
    if jp.exists():
        shutil.rmtree(jp, ignore_errors=True)
        print(f'Da xoa sach thu muc: {jp.name}/')

# 4. In bao cao danh sach file con lai trong /kaggle/working/
print('\n' + '=' * 70)
print('DANH SÁCH FILE ĐẦU RA SẠCH DUY NHẤT CÒN LẠI TRÊN /kaggle/working/ :')
print('=' * 70)
total_size = 0
for item in sorted(WORKING_DIR.glob('*')):
    if item.is_file():
        sz = item.stat().st_size
        total_size += sz
        sz_str = f'{sz/1024/1024:.2f} MB' if sz > 1024*1024 else f'{sz/1024:.1f} KB'
        print(f'  - {item.name:<45} | Size: {sz_str}')
print(f'TONG DUNG LUONG DAU RA: {total_size/1024/1024:.2f} MB')
print('>>> QUA TRINH HUAN LUYEN VA DON DEP DA HOAN TAT AN TOAN 100%! <<<')